# M3 Split/Causality PI Response

Purpose: run the experimental-design check requested before M3.3 feature work. This notebook reuses the M3.2 pipeline: 17 baseline features plus value-change features, LightGBM only, `random_state=42`, and downsampling seeds `10/20`.

Axes:
- Building split: existing `building_id % 5 == 4` 80/20, and PI protocol `building_id % 2` 50/50.
- Value-change regime: offline uses past + future shifts; causal uses positive/past shifts only.

Primary output: `data/processed/m3_split_causality_results.json`.

In [ ]:
import json
from pathlib import Path

ROOT = Path("..").resolve()
RESULTS_PATH = ROOT / "data" / "processed" / "m3_split_causality_results.json"
results = json.loads(RESULTS_PATH.read_text())

print(f"random_state={results['random_state']}")
print(f"downsampling_seeds={results['downsampling_seeds']}")
print(f"shifts={len(results['shift_set'])}: {results['shift_set']}")
print(f"past_shifts={len(results['past_shifts'])}: {results['past_shifts']}")
print(f"future_shifts={len(results['future_shifts'])}: {results['future_shifts']}")

## 2x2 Result Grid

| Split | Regime | Features | Train buildings | Val buildings | Train anomaly | Val anomaly | Val AUC | Precision@0.5 | Recall@0.5 | F1@0.5 |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 80/20 mod5 | offline | 137 | 1160 | 289 | 6.65% | 5.93% | 0.9920 | 0.6409 | 0.9665 | 0.7707 |
| 80/20 mod5 | causal | 77 | 1160 | 289 | 6.65% | 5.93% | 0.9908 | 0.6237 | 0.9603 | 0.7562 |
| 50/50 mod2 | offline | 137 | 725 | 724 | 6.72% | 6.29% | 0.9914 | 0.6878 | 0.9421 | 0.7951 |
| 50/50 mod2 | causal | 77 | 725 | 724 | 6.72% | 6.29% | 0.9903 | 0.6646 | 0.9355 | 0.7772 |

Notes:
- 80/20-offline reproduces M3.2 (`AUC=0.9920`).
- The 50/50 AUC dip is the cost of the PI split protocol: fewer train buildings, not a feature regression.
- The causal AUC dip is the cost of real-time deployability: the model can no longer use future meter readings.
- Causal operationalizes the M3.2 past/future leakage check by keeping only the past contribution.

In [ ]:
rows = []
for split_name, split_results in results["splits"].items():
    for regime, m in split_results.items():
        rows.append(
            {
                "split": split_name,
                "regime": regime,
                "features": m["n_features"],
                "train_buildings": m["n_train_buildings"],
                "val_buildings": m["n_val_buildings"],
                "train_anomaly_rate": round(m["train_anomaly_rate"], 4),
                "val_anomaly_rate": round(m["val_anomaly_rate"], 4),
                "val_auc": round(m["val_auc"], 4),
                "precision_05": round(m["precision_05"], 4),
                "recall_05": round(m["recall_05"], 4),
                "f1_05": round(m["f1_05"], 4),
                "building_overlap": m["building_overlap"],
            }
        )
rows

## Robustness and Sanity

Seeded-random 50/50 building split (`random_state=42`, offline): AUC `0.9910`, close to deterministic 50/50 offline AUC `0.9914` (delta `-0.0004`).

Label-shuffle sanity check on 50/50 causal: AUC `0.4527`, confirming the causal result is not produced by split leakage or spurious label alignment.

All reported splits have `building_overlap=0`.

In [ ]:
robust = results["robustness_checks"]["50_50_random42"]
shuffle = results["sanity_checks"]["50_50_mod2_causal_label_shuffle"]
print("random 50/50 offline AUC:", round(robust["val_auc"], 4))
print(
    "mod2 50/50 offline AUC:",
    round(results["splits"]["50_50_mod2"]["offline"]["val_auc"], 4),
)
print("label-shuffle 50/50 causal AUC:", round(shuffle["val_auc"], 4))

## Re-run Command

The experiment was run with:

```powershell
$env:UV_CACHE_DIR='C:\\Users\\tonykuo\\projects\\lead-reproduction\\.uv-cache'
$env:UV_PYTHON_INSTALL_DIR='C:\\Users\\tonykuo\\projects\\lead-reproduction\\.uv-python'
uv run python .scratch\\run_m3_split_causality.py
```

The runner is intentionally separate from M3.3 feature work and does not add new features.